# Topic 3 — Model Context Protocol (MCP): Practice Notebook

This notebook is the hands-on companion to `notes/03-model-context-protocol-mcp.md`. Read that note first — it explains the *why* behind the protocol, the JSON-RPC message shapes, and the security model, with full dry runs. This notebook focuses on running the code yourself.

**How this notebook stays offline and deterministic**: the MCP server (`mcp_server.py`, in this same folder) is a small SQLite-backed server that runs entirely on your machine — no API key, no network call, no external service. The agent in Section 4 uses `GenericFakeChatModel` from `langchain_core`, exactly as in Topic 2.

**Sections**:

1. Protocol Basics
2. Building a Minimal MCP Server
3. Running & Inspecting the Server
4. Connecting a Client — **EXERCISE**: `build_mcp_agent_graph`, and **EXERCISE**: `add_note` in `mcp_server.py`
5. Transports
6. Security Model

Cells marked **EXERCISE** contain a function (or, for `add_note`, a tool inside `mcp_server.py`) with a docstring describing exactly what to implement, followed by `# YOUR CODE HERE`. The `solutions/` copy of this notebook (and `mcp_server.py`) starts identical to this one — work through it there, and check `solutions/03-mcp-code-explanation.md` for the canonical implementations, full walkthroughs, and dry runs once you're done.

**Note**: this notebook spawns `mcp_server.py` as a subprocess, so it must be run from this folder (`code/template/`).

In [ ]:
import json
from pathlib import Path
from typing import TypedDict, Annotated

from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

import mcp.types as types
from mcp import ClientSession
from mcp.client.stdio import StdioServerParameters, stdio_client
from mcp.server.fastmcp import FastMCP

from langchain_mcp_adapters.client import MultiServerMCPClient

print("Imports OK. mcp_server.py (in this folder) is the standalone MCP server we'll talk to.")

## 1. Protocol Basics

MCP is a **client/server** protocol: every message between them is a **JSON-RPC 2.0** message — a request `{jsonrpc, id, method, params}` answered by a response `{jsonrpc, id, result}` (or `{jsonrpc, id, error}`) carrying the *same* `id`. A server can expose three kinds of capability:

- **tools** — actions with side effects, discovered via `tools/list` and invoked via `tools/call`. This is the cross-process generalization of Topic 1/2's `@tool`.
- **resources** — read-only data identified by a URI (e.g. `notes://topics`), discovered via `resources/list` and fetched via `resources/read`.
- **prompts** — reusable, parameterized prompt templates the server exposes, discovered via `prompts/list` and filled via `prompts/get`.

The cell below constructs one example of each primitive directly from `mcp.types` and prints its JSON shape — this is what a server *describes itself with*, before any server is even running.

In [ ]:
tool_description = types.Tool(
    name="search_notes",
    description="Search the learning notes database for a topic and return its content.",
    inputSchema={"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
)

resource_description = types.Resource(
    name="topics",
    uri="notes://topics",
    description="List every topic available in the learning notes database.",
    mimeType="text/plain",
)

prompt_description = types.Prompt(
    name="summarize_note",
    description="Summarize a learning note in one sentence.",
    arguments=[types.PromptArgument(name="topic", description="The topic to summarize", required=True)],
)

print("TOOL primitive:")
print(json.dumps(tool_description.model_dump(exclude_none=True), indent=2))

print("\nRESOURCE primitive:")
print(json.dumps(resource_description.model_dump(exclude_none=True, mode="json"), indent=2))

print("\nPROMPT primitive:")
print(json.dumps(prompt_description.model_dump(exclude_none=True), indent=2))

## 2. Building a Minimal MCP Server

`FastMCP` (from the official `mcp` Python SDK) turns plain Python functions into MCP tools and resources via decorators — `@mcp.tool()` and `@mcp.resource(uri)` — deriving each tool's JSON Schema from its type hints, the same mechanism Topic 1 used for `@tool` / `bind_tools`.

The cell below builds a tiny `FastMCP` server *in-process* (no subprocess, no transport yet) with one tool, and introspects it directly — `FastMCP` objects can answer `list_tools()` / `call_tool()` calls without running a transport loop.

In [ ]:
demo_mcp = FastMCP("demo")


@demo_mcp.tool()
def add_numbers(a: int, b: int) -> int:
    """Add two integers together."""
    return a + b


demo_tools = await demo_mcp.list_tools()
for demo_tool in demo_tools:
    print("name:", demo_tool.name)
    print("description:", demo_tool.description)
    print("input_schema:", demo_tool.inputSchema)

content, structured = await demo_mcp.call_tool("add_numbers", {"a": 3, "b": 4})
print("\ncall_tool result:")
print("content:", content)
print("structured:", structured)

The *real* server for this notebook, `mcp_server.py`, lives in this same folder as a standalone file — because MCP servers run as **separate processes** (Section 3 spawns this file as a subprocess), it can't be a notebook cell. It seeds a small SQLite database (`learning_notes.db`) with one-sentence notes (reusing several topics from Topic 2's `SEARCH_INDEX`), then exposes:

- `search_notes(query: str) -> str` — a **tool** (read-only) that looks up a note by topic.
- `notes://topics` — a **resource** listing every topic in the database.
- `add_note(topic: str, content: str) -> str` — a second **tool** (destructive) that writes a new note. **This is Exercise 2** (Section 4) — its body currently raises `NotImplementedError`.

Run the cell below to print its full source.

In [ ]:
print(Path("mcp_server.py").read_text())

## 3. Running & Inspecting the Server

A client connects to a server process via a **transport** — here, **stdio**: `StdioServerParameters(command="python3", args=["mcp_server.py"])` describes how to start the server, `stdio_client(...)` spawns it and gives you its stdin/stdout as two streams, and `ClientSession(read, write)` wraps those streams with the MCP protocol. `await session.initialize()` performs the handshake; after that, `list_tools`, `list_resources`, `call_tool`, and `read_resource` are all available.

In [ ]:
server_params = StdioServerParameters(command="python3", args=["mcp_server.py"])

async with stdio_client(server_params) as (read_stream, write_stream):
    async with ClientSession(read_stream, write_stream) as session:
        init_result = await session.initialize()
        print("server name:", init_result.serverInfo.name)
        print("protocol version:", init_result.protocolVersion)

        print("\n--- list_tools ---")
        tools_result = await session.list_tools()
        for tool in tools_result.tools:
            print("name:", tool.name)
            print("description:", tool.description)

        print("\n--- list_resources ---")
        resources_result = await session.list_resources()
        for resource in resources_result.resources:
            print("uri:", str(resource.uri))
            print("name:", resource.name)
            print("description:", resource.description)

        print("\n--- call_tool: search_notes(query='langgraph') ---")
        result = await session.call_tool("search_notes", {"query": "langgraph"})
        print("isError:", result.isError)
        for block in result.content:
            print("content:", block.text)

        print("\n--- call_tool: search_notes(query='quantum gravity') ---")
        result = await session.call_tool("search_notes", {"query": "quantum gravity"})
        for block in result.content:
            print("content:", block.text)

        print("\n--- read_resource: notes://topics ---")
        read_result = await session.read_resource("notes://topics")
        for resource_content in read_result.contents:
            print("text:", resource_content.text)

### Raw JSON-RPC for one `tools/call`

Every `session.call_tool(...)` above sent a JSON-RPC request over stdin and read a JSON-RPC response from stdout. The cell below shows that envelope explicitly for `search_notes(query="react")`: the request `params` carry the tool name and arguments, and the response `result.content` is a **list** of typed content blocks (here, one `{"type": "text", ...}` block) plus an `isError` flag — richer than the plain string Topic 2's in-process `web_search` returned directly.

In [ ]:
request = {
    "jsonrpc": "2.0",
    "id": 99,
    "method": "tools/call",
    "params": {"name": "search_notes", "arguments": {"query": "react"}},
}
print("request:")
print(json.dumps(request, indent=2))

server_params = StdioServerParameters(command="python3", args=["mcp_server.py"])
async with stdio_client(server_params) as (read_stream, write_stream):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()
        result = await session.call_tool("search_notes", {"query": "react"})

response = {
    "jsonrpc": "2.0",
    "id": 99,
    "result": {
        "content": [{"type": block.type, "text": block.text} for block in result.content],
        "isError": result.isError,
    },
}
print("\nresponse:")
print(json.dumps(response, indent=2))

## 4. Connecting a Client

`MultiServerMCPClient` (from `langchain-mcp-adapters`) connects to one or more MCP servers and exposes their tools as ordinary LangChain `BaseTool` objects via `await client.get_tools()`. Each returned tool's `.ainvoke()` performs a `tools/call` round trip to the server under the hood — so it drops into a LangGraph `ToolNode` exactly like Topic 2's `web_search`, even though `search_notes` now runs in a separate process (`mcp_server.py`).

In [ ]:
client = MultiServerMCPClient({
    "learning_notes": {
        "transport": "stdio",
        "command": "python3",
        "args": ["mcp_server.py"],
    }
})

mcp_tools = await client.get_tools()
for mcp_tool in mcp_tools:
    print("tool:", mcp_tool.name, "-", mcp_tool.description)

### Exercise 1 — `build_mcp_agent_graph`

Build the same agent/tools cycle as Topic 2 Section 3 (`AgentState`, `agent_node`, `StateGraph`, `ToolNode`, `tools_condition`), but parameterized over the MCP-loaded `tools` list and an `llm` — so it can be reused with any set of tools, MCP-backed or not.

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


def build_mcp_agent_graph(tools, llm):
    """Build a 2-node agent/tools LangGraph app using MCP-loaded tools.

    1. Define agent_node(state) that returns
       {"messages": [llm.invoke(state["messages"])]}.
    2. Build a StateGraph(AgentState): add "agent" (agent_node) and
       "tools" (ToolNode(tools)) nodes.
    3. Wire START -> "agent", "agent" -> (tools_condition) -> "tools" or END,
       and "tools" -> "agent".
    4. Compile and return the app.
    """
    # YOUR CODE HERE
    pass

In [ ]:
agent_llm = GenericFakeChatModel(messages=iter([
    AIMessage(content="", tool_calls=[{"name": "search_notes", "args": {"query": "mcp"}, "id": "call_1"}]),
    AIMessage(content="MCP standardizes how LLM apps access external tools, data, and prompts."),
]))

mcp_agent_app = build_mcp_agent_graph(mcp_tools, agent_llm)

if mcp_agent_app is None:
    print("Exercise not yet solved -- build_mcp_agent_graph returned None, skipping the agent run.")
else:
    result = await mcp_agent_app.ainvoke({"messages": [HumanMessage(content="What is MCP?")]})
    for message in result["messages"]:
        print(type(message).__name__, "|", repr(message.content))

### Exercise 2 — `add_note` (in `mcp_server.py`)

Open `mcp_server.py` and implement the body of `add_note`, following its docstring. Once implemented, the cell below calls `add_note(topic="transports", content="...")` and then `search_notes(query="transports")` — demonstrating that, unlike Topic 2's fixed `SEARCH_INDEX` dict, this MCP server has **state that persists** across calls (and across separate client connections), backed by `learning_notes.db`.

If `add_note` is still unimplemented, the call returns `isError=True` with a descriptive message instead of crashing — `FastMCP` converts an unhandled exception inside a tool into an MCP error response.

In [ ]:
client_2 = MultiServerMCPClient({
    "learning_notes": {
        "transport": "stdio",
        "command": "python3",
        "args": ["mcp_server.py"],
    }
})

async with client_2.session("learning_notes") as session:
    result = await session.call_tool(
        "add_note",
        {"topic": "transports", "content": "stdio is for local processes; streamable HTTP is for remote/shared servers."},
    )
    print("isError:", result.isError)
    for block in result.content:
        print("content:", block.text)

    if not result.isError:
        verify = await session.call_tool("search_notes", {"query": "transports"})
        for block in verify.content:
            print("verify:", block.text)
    else:
        print("\nadd_note is not implemented yet -- nothing to verify.")

## 5. Transports

stdio (used throughout this notebook) is like plugging a USB peripheral directly into your laptop: the client spawns the server as a subprocess, one client per server, and the server only exists while something is using it. **Streamable HTTP** (the modern replacement for the older HTTP+SSE transport) is like a network printer: the server runs independently at a host:port, and any client that can reach it can connect — at the cost of needing an address and (per Section 6) authentication.

| | stdio | Streamable HTTP |
|---|---|---|
| Starts | Client spawns server subprocess | Server runs independently |
| Connects | Only the spawning process | Any client with the URL |
| Lifetime | Tied to the client | Independent |
| Typical use | Local dev tools (Claude Code's local MCP servers) | Remote/shared services |

The only thing that changes between transports is the last line of `mcp_server.py` and the client's connection config — everything else (`@mcp.tool()`, `@mcp.resource()`, `ClientSession.call_tool()`) is identical.

In [ ]:
stdio_server_line = 'mcp.run(transport="stdio")'
http_server_line = 'mcp.run(transport="streamable-http", host="0.0.0.0", port=8000)'

stdio_client_config = {"transport": "stdio", "command": "python3", "args": ["mcp_server.py"]}
http_client_config = {"transport": "streamable_http", "url": "http://localhost:8000/mcp"}

print("Server-side (mcp_server.py), last line only:")
print("  stdio:          ", stdio_server_line)
print("  streamable HTTP:", http_server_line)

print("\nClient-side connection config:")
print("  stdio:          ", stdio_client_config)
print("  streamable HTTP:", http_client_config)

## 6. Security Model

`ToolAnnotations` let a server declare a tool's risk profile — `readOnlyHint`, `destructiveHint`, `idempotentHint`, `openWorldHint` — so a client can decide when to require explicit user consent, the same way Claude Code silently runs `git status` but prompts before `Edit` or an unrecognized `Bash` command. `mcp_server.py` marks `search_notes` as `readOnlyHint=True, openWorldHint=False` and `add_note` as `destructiveHint=True, idempotentHint=False`. The cell below re-fetches `list_tools()` and prints these annotations directly.

These are **hints**, not enforcement — the real security boundary is the client's consent policy and, ultimately, the user approving (or denying) the action when prompted. This is the tool-call-layer counterpart to Topic 2's `interrupt()` / `Command(resume=...)` human-in-the-loop pattern.

In [ ]:
server_params = StdioServerParameters(command="python3", args=["mcp_server.py"])
async with stdio_client(server_params) as (read_stream, write_stream):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()
        tools_result = await session.list_tools()
        for tool in tools_result.tools:
            print(tool.name, "->", tool.annotations)

## Putting It All Together

```
notebook (client)                            mcp_server.py (separate process)
  |                                                |
  |-- initialize -------------------------------->|
  |<----------------------- serverInfo -----------|
  |-- tools/list --------------------------------->|
  |<--------- [search_notes, add_note] -----------|
  |-- tools/call(search_notes, {query}) --------->|
  |<------------------ result.content ------------|   SQLite: learning_notes.db
  |-- resources/read(notes://topics) ------------>|
  |<------------------ topic list -----------------|
  |                                                |
  |== via langchain-mcp-adapters ==>  ToolNode([search_notes, add_note])
  |                                       |
  |                              agent <-> tools  (Topic 2's cycle, unchanged)
```

Section 1 gave the message format and the three primitives. Section 2 built the server with `FastMCP` decorators. Section 3 ran it over stdio and inspected the raw protocol. Section 4 wrapped its tools as LangChain `BaseTool`s and dropped them into a Topic-2-style agent graph — plus extended the server itself with a stateful `add_note` tool. Sections 5-6 stepped back to the operational concerns (transport choice, consent) that matter once this server leaves your laptop.

## Where to Go Next

Topic 4 (AI Agent Patterns & Frameworks) takes the pieces from Topics 2-3 — a stateful graph, an MCP-backed tool server, human-in-the-loop approval — and compares this hand-built approach against higher-level agent frameworks (LangGraph's own prebuilt agents, CrewAI, the Claude Agent SDK), which package the same primitives behind different APIs.